# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [1]:
ПРІЗВИЩЕ_ІМЯ = 'Бобринок Ангеліна'      # напишіть тут своє прізвище та ім'я
ГРУПА        = 'ШІ-32'
НОМЕР        = 3       # ваш номер за списком групи (1–30)

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [2]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', None)   # повний текст у таблицях (для PDF)

ВАРІАНТИ = json.load(open('lr3_variants_student.json'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 3   ·   Бобринок Ангеліна   ·   ШІ-32

Ваші вісім стовпців:
   1. номер і/х
   2. помер/жив
   3. Перебіг
   4. Виписка
   5. spO2 без дот
   6. Креат1
   7. Гемогл2
   8. БРзаг2

Ваша пара для аналізу зв’язку: Перебіг та ступінь КТ

Ваше додаткове питання:
   Стовпець «помер/жив» містить числа 1 і 2. Які три різні статистики pandas порахує для нього без жодної скарги — і чому кожна з них безглузда?


In [3]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,номер і/х,помер/жив,Перебіг,Виписка,spO2 без дот,Креат1,Гемогл2,БРзаг2
0,2377/239,1,важкий,2022-03-10 00:00:00,90.0,84,144,17.0
1,1121/19,2,важкий,2022-03-16 00:00:00,83.0,75.4,137,13.6
2,283/674,2,важкий,2022-01-18 00:00:00,78.0,80.7,105,18.7
3,2372/513,2,важкий,2022-01-13 00:00:00,82.0,168.1,116,13.6
4,1441/671,2,важкий,2022-03-11 00:00:00,80.0,76,124,15.3
5,2573/354,1,важкий,2021-12-26 00:00:00,72.0,130,120,15.3
6,1762/555,2,важкий,2022-05-11 00:00:00,89.0,98,151,15.3
7,2664/348,2,серед важ,2022-05-28 00:00:00,90.0,118,137,15.3


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [4]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   номер і/х     121 non-null    object 
 1   помер/жив     121 non-null    int64  
 2   Перебіг       121 non-null    str    
 3   Виписка       120 non-null    object 
 4   spO2 без дот  120 non-null    float64
 5   Креат1        117 non-null    object 
 6   Гемогл2       105 non-null    object 
 7   БРзаг2        101 non-null    float64
dtypes: float64(2), int64(1), object(4), str(1)
memory usage: 9.3+ KB


In [5]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- номер і/х
    непорожніх 121 з 121 | унікальних 119 | типи: int×65, str×56
    приклади: ["'2377/239'", "'1121/19'", "'283/674'", "'2372/513'"]

--- помер/жив
    непорожніх 121 з 121 | унікальних 2 | типи: int×121
    приклади: ['np.int64(1)', 'np.int64(2)']

--- Перебіг
    непорожніх 121 з 121 | унікальних 3 | типи: str×121
    приклади: ["'важкий'", "'серед важ'", "'вкрай важкий '"]

--- Виписка
    непорожніх 120 з 121 | унікальних 90 | типи: datetime×118, str×2
    приклади: ['datetime.datetime(2022, 3, 10, 0, 0)', 'datetime.datetime(2022, 3, 16, 0, 0)', 'datetime.datetime(2022, 1, 18, 0, 0)', 'datetime.datetime(2022, 1, 13, 0, 0)']

--- spO2 без дот
    непорожніх 120 з 121 | унікальних 33 | типи: float×120
    приклади: ['np.float64(90.0)', 'np.float64(83.0)', 'np.float64(78.0)', 'np.float64(82.0)']

--- Креат1
    непорожніх 117 з 121 | унікальних 73 | типи: int×104, float×11, str×2
    приклади: ['84', '75.4', '80.7', '168.1']

--- Гемогл2
    непорожніх 105 з 121 | уніка

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [6]:
ПАСПОРТ = {
    'номер і/х':    ('ідентифікатор',
                     'Номер історії хвороби лише позначає запис: формат змішаний (1837 і 2377/239), '
                     'сума чи різниця номерів нічого не означає, і навіть рівність ненадійна — '
                     'номери 1837 та 2526 повторюються у двох різних пацієнтів.'),
    'помер/жив':    ('номінальна',
                     'Це бінарна категорія результату (1 = помер, 2 = вижив); цифри 1 і 2 — лише '
                     'мітки, їх можна замінити на «так/ні», тому значення порівнюються тільки на рівність.'),
    'Перебіг':      ('порядкова',
                     'Рівні «серед важ» < «важкий» < «вкрай важкий» мають природний порядок тяжкості, '
                     'але відстань між сусідніми рівнями невідома й не обов’язково однакова.'),
    'Виписка':      ('інтервальна',
                     'Дата календаря: різниця двох дат (наприклад, ліжко-дні) має сенс, а нуль '
                     'умовний — «удвічі пізніша дата» не означає нічого.'),
    'spO2 без дот': ('абсолютна',
                     'Сатурація — частка гемоглобіну, насиченого киснем, у відсотках: є природний '
                     'нуль (0 %) і природна одиниця/межа (100 %), тож відношення й частки мають сенс.'),
    'Креат1':       ('відношень',
                     'Концентрація креатиніну в мкмоль/л: нуль означає відсутність речовини, тому '
                     '«креатинін удвічі вищий» — змістовне твердження; одиниця вимірювання умовна.'),
    'Гемогл2':      ('відношень',
                     'Концентрація гемоглобіну в г/л має абсолютний нуль (немає гемоглобіну), '
                     'відношення 140/70 = 2 має фізичний зміст, але одиницю (г/л чи г/дл) обрано умовно.'),
    'БРзаг2':       ('відношень',
                     'Загальний білірубін у мкмоль/л — концентрація з природним нулем, допустимі '
                     'відношення («білірубін утричі вищий за норму»), одиниця умовна.'),
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,номер і/х,ідентифікатор,"Номер історії хвороби лише позначає запис: формат змішаний (1837 і 2377/239), сума чи різниця номерів нічого не означає, і навіть рівність ненадійна — номери 1837 та 2526 повторюються у двох різних пацієнтів."
1,помер/жив,номінальна,"Це бінарна категорія результату (1 = помер, 2 = вижив); цифри 1 і 2 — лише мітки, їх можна замінити на «так/ні», тому значення порівнюються тільки на рівність."
2,Перебіг,порядкова,"Рівні «серед важ» < «важкий» < «вкрай важкий» мають природний порядок тяжкості, але відстань між сусідніми рівнями невідома й не обов’язково однакова."
3,Виписка,інтервальна,"Дата календаря: різниця двох дат (наприклад, ліжко-дні) має сенс, а нуль умовний — «удвічі пізніша дата» не означає нічого."
4,spO2 без дот,абсолютна,"Сатурація — частка гемоглобіну, насиченого киснем, у відсотках: є природний нуль (0 %) і природна одиниця/межа (100 %), тож відношення й частки мають сенс."
5,Креат1,відношень,"Концентрація креатиніну в мкмоль/л: нуль означає відсутність речовини, тому «креатинін удвічі вищий» — змістовне твердження; одиниця вимірювання умовна."
6,Гемогл2,відношень,"Концентрація гемоглобіну в г/л має абсолютний нуль (немає гемоглобіну), відношення 140/70 = 2 має фізичний зміст, але одиницю (г/л чи г/дл) обрано умовно."
7,БРзаг2,відношень,"Загальний білірубін у мкмоль/л — концентрація з природним нулем, допустимі відношення («білірубін утричі вищий за норму»), одиниця умовна."


In [7]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

### Підготовка значень до обчислень

Перш ніж рахувати, треба привести значення до того типу, якого вимагає шкала.
У моїх стовпцях є дефекти введення:

* `Креат1` — рядки `'109.9'` (число, записане текстом) і `'даних'` (слово замість пропуску);
* `Гемогл2` — рядок `'немає'`;
* `Виписка` — дві неіснуючі дати `'24.024.2021'` і `'20.24.2021'`;
* `spO2 без дот` — значення `0.9`: сатурацію записано часткою, а не відсотками (90 %);
* `Перебіг` — `'вкрай важкий '` із пробілом у кінці.

In [8]:
ПОРЯДОК_ПЕРЕБІГ = ['серед важ', 'важкий', 'вкрай важкий']   # порядок задано вручну


def очищений(стовпець):
    """Повертає стовпець, приведений до типу, якого вимагає його шкала."""
    s = df[стовпець]
    if стовпець == 'Перебіг':
        return pd.Categorical(s.str.strip(), categories=ПОРЯДОК_ПЕРЕБІГ, ordered=True)
    if стовпець == 'Виписка':
        return pd.to_datetime(s, errors='coerce')          # '24.024.2021' -> NaT
    if стовпець == 'spO2 без дот':
        s = pd.to_numeric(s, errors='coerce')
        return s.where(s > 1, s * 100)                     # 0.9 -> 90 %
    if ПАСПОРТ[стовпець][0] in ('відношень', 'абсолютна'):
        return pd.to_numeric(s, errors='coerce')           # 'даних', 'немає' -> NaN
    return s


for c in ['Креат1', 'Гемогл2', 'Виписка', 'spO2 без дот']:
    сире = df[c].notna().sum()
    чисте = pd.Series(очищений(c)).notna().sum()
    print(f'{c:14s} непорожніх до очищення: {сире:3d}, після: {чисте:3d}')

Креат1         непорожніх до очищення: 117, після: 116
Гемогл2        непорожніх до очищення: 105, після: 104
Виписка        непорожніх до очищення: 120, після: 118
spO2 без дот   непорожніх до очищення: 120, після: 120


In [9]:
def допустима_статистика(стовпець, шкала):
    """Повертає словник статистик, дозволених для цієї шкали."""
    з = pd.Series(очищений(стовпець)).dropna()
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()

    if ш in ('ідентифікатор',):
        return р

    # усі шкали: мода й частоти
    частоти = з.value_counts()
    р['мода'] = частоти.index[0]
    р['частоти (топ-3)'] = частоти.head(3).to_dict()

    if ш == 'номінальна':
        return р

    # порядкова і вище: медіана та квартилі
    if ш == 'порядкова':
        коди = з.cat.codes                              # коди за заданим порядком
        р['медіана'] = з.cat.categories[int(коди.median())]
        р['Q1'] = з.cat.categories[int(коди.quantile(.25, interpolation='lower'))]
        р['Q3'] = з.cat.categories[int(коди.quantile(.75, interpolation='higher'))]
        return р

    р['медіана'] = з.median()
    р['Q1'], р['Q3'] = з.quantile(.25), з.quantile(.75)

    # інтервальна і вище: середнє, стандартне відхилення, різниці
    р['середнє'] = з.mean()
    р['стд. відхилення'] = з.std()
    р['розмах (max − min)'] = з.max() - з.min()
    if ш == 'інтервальна':
        return р

    # відношень і абсолютна: коефіцієнт варіації, геометричне середнє
    р['коеф. варіації, %'] = 100 * з.std() / з.mean()
    р['геом. середнє'] = stats.gmean(з[з > 0])
    р['max / min'] = з.max() / з.min()
    return р


for c in МОЇ:
    print(f'--- {c}  ({ПАСПОРТ[c][0]})')
    for k, v in допустима_статистика(c, ПАСПОРТ[c][0]).items():
        if isinstance(v, float):
            v = round(v, 2)
        print(f'    {k:20s}: {v}')
    print()

--- номер і/х  (ідентифікатор)
    непорожніх          : 121
    унікальних          : 119

--- помер/жив  (номінальна)
    непорожніх          : 121
    унікальних          : 2
    мода                : 1
    частоти (топ-3)     : {1: 61, 2: 60}

--- Перебіг  (порядкова)
    непорожніх          : 121
    унікальних          : 3
    мода                : важкий
    частоти (топ-3)     : {'важкий': 90, 'серед важ': 30, 'вкрай важкий': 1}
    медіана             : важкий
    Q1                  : важкий
    Q3                  : важкий

--- Виписка  (інтервальна)
    непорожніх          : 118
    унікальних          : 88
    мода                : 2021-11-19 00:00:00
    частоти (топ-3)     : {Timestamp('2021-11-19 00:00:00'): 3, Timestamp('2021-09-07 00:00:00'): 3, Timestamp('2021-09-15 00:00:00'): 3}
    медіана             : 2022-03-15 00:00:00
    Q1                  : 2021-11-16 18:00:00
    Q3                  : 2022-04-27 18:00:00
    середнє             : 2022-01-29 18:06:06.10169

**Що недопустимо для кожного стовпця:**

* `номер і/х` — нічого, крім кількості та унікальності; навіть мода (1837) показує не
  «типовий номер», а помилку введення — два різних пацієнти з одним номером;
* `помер/жив` — лише частоти та мода; медіана, середнє й стд. відхилення залежать від того,
  якими цифрами закодовано категорії;
* `Перебіг` — медіана і квартилі за **заданим** порядком; середнє недопустиме, бо відстані
  між «серед важ», «важкий» і «вкрай важкий» невідомі;
* `Виписка` — середнє, стд. і різниці дат; коефіцієнт варіації чи «max / min» недопустимі,
  бо нуль дати (1970-01-01 у pandas) умовний;
* `spO2 без дот`, `Креат1`, `Гемогл2`, `БРзаг2` — усі статистики, включно з CV і геометричним середнім.

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [10]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
номер і/х,121.0,119.0,1837.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
помер/жив,121.0,NaN,NaN,NaN,1.495868,0.502062,1.0,1.0,1.0,2.0,2.0
Перебіг,121,3,важкий,90,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Виписка,120,90,2021-11-19 00:00:00,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
spO2 без дот,120.0,NaN,NaN,NaN,82.049167,12.208249,0.9,78.75,84.0,89.0,98.0
Креат1,117.0,73.0,76.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Гемогл2,105.0,60.0,144.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
БРзаг2,101.0,NaN,NaN,NaN,16.216832,5.537094,7.4,13.6,15.3,18.7,60.5


In [11]:
номер_як_число = pd.to_numeric(df['номер і/х'], errors='coerce')
пж = df['помер/жив']

БЕЗГЛУЗДІ = [
    ('помер/жив', 'mean', round(пж.mean(), 3),
     'Середнє номінальних міток: «1.496» не є жодним результатом лікування; якщо закодувати '
     'помер=0, вижив=1, вийде 0.496, а при 1/0 — 0.504, тобто число залежить від довільного кодування.'),
    ('помер/жив', 'std', round(пж.std(), 3),
     'Стандартне відхилення відстаней між мітками 1 і 2 — відстаней між «помер» і «вижив» '
     'не існує, тому і розкиду навколо «середнього результату» немає.'),
    ('помер/жив', '50% (медіана)', пж.median(),
     'Медіана вимагає порядку; «медіанний пацієнт помер» — артефакт того, що смерть закодовано '
     'меншою цифрою (61 проти 60). Поміняйте коди — і медіана «оживе».'),
    ('spO2 без дот', 'min', df['spO2 без дот'].min(),
     'Сатурація 0.9 % несумісна з життям — це 90 %, записані часткою. Число арифметично правильне, '
     'але дані непридатні: воно також тягне вниз середнє (82.05 замість 82.79) і роздуває std.'),
    ('номер і/х', 'mean (числових номерів)', round(номер_як_число.mean(), 1),
     'Середнє ідентифікаторів — це «середній номер історії хвороби», який не належить жодному '
     'пацієнту й нічого не вимірює; до того ж половину номерів (формат «2377/239») відкинуто.'),
    ('номер і/х', 'top (мода)', df['номер і/х'].mode()[0],
     'describe() показує «найчастіший номер», але ідентифікатор має бути унікальним; '
     'freq = 2 вказує не на типове значення, а на дублікат/помилку введення.'),
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,помер/жив,mean,1.496,"Середнє номінальних міток: «1.496» не є жодним результатом лікування; якщо закодувати помер=0, вижив=1, вийде 0.496, а при 1/0 — 0.504, тобто число залежить від довільного кодування."
1,помер/жив,std,0.502,"Стандартне відхилення відстаней між мітками 1 і 2 — відстаней між «помер» і «вижив» не існує, тому і розкиду навколо «середнього результату» немає."
2,помер/жив,50% (медіана),1.000,"Медіана вимагає порядку; «медіанний пацієнт помер» — артефакт того, що смерть закодовано меншою цифрою (61 проти 60). Поміняйте коди — і медіана «оживе»."
3,spO2 без дот,min,0.900,"Сатурація 0.9 % несумісна з життям — це 90 %, записані часткою. Число арифметично правильне, але дані непридатні: воно також тягне вниз середнє (82.05 замість 82.79) і роздуває std."
4,номер і/х,mean (числових номерів),1770.100,"Середнє ідентифікаторів — це «середній номер історії хвороби», який не належить жодному пацієнту й нічого не вимірює; до того ж половину номерів (формат «2377/239») відкинуто."
5,номер і/х,top (мода),1837.000,"describe() показує «найчастіший номер», але ідентифікатор має бути унікальним; freq = 2 вказує не на типове значення, а на дублікат/помилку введення."


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [12]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала — порядкова (визначено самостійно: легкий < середній < важкий < вкрай важкий < критичний)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: Перебіг × ступінь КТ

Перебіг: шкала — порядкова
ступінь КТ: шкала — порядкова (визначено самостійно: легкий < середній < важкий < вкрай важкий < критичний)

  Перебіг ступінь КТ
0  важкий    сердній
1  важкий   середній
2  важкий  критичний
3  важкий  критичний
4  важкий        NaN
5  важкий   середній


In [13]:
# Обидва стовпці — порядкові, але записані «як вийде». Нормалізую написання
# і задаю порядок ВРУЧНУ: алфавітне сортування ставить «важкий» перед «легкий».
print('ступінь КТ до очищення:', sorted(df[B].dropna().map(repr).unique()))

ПОРЯДОК_КТ = ['легкий', 'середній', 'важкий', 'вкрай важкий', 'критичний']
СИНОНІМИ_КТ = {'легка': 'легкий', 'сердній': 'середній',
               'серед ступ': 'середній', 'серед ступеня': 'середній'}

кт_текст = df[B].astype('string').str.strip().str.lower().replace(СИНОНІМИ_КТ)
кт = pd.Series(pd.Categorical(кт_текст, categories=ПОРЯДОК_КТ, ordered=True), index=df.index)
перебіг = pd.Series(очищений(A), index=df.index)

пара = pd.DataFrame({A: перебіг, B: кт}).dropna()
print('ступінь КТ після очищення:', list(кт.value_counts().items()))
print(f'Пар без пропусків: {len(пара)} з {len(df)} (ступінь КТ заповнено лише у {кт.notna().sum()} пацієнтів)')
print()
pd.crosstab(пара[A], пара[B])

ступінь КТ до очищення: ["'важкий '", "'важкий'", "'вкрай важкий '", "'вкрай важкий'", "'критичний'", "'легка'", "'легкий '", "'легкий'", "'сердній'", "'серед ступ'", "'серед ступеня'", "'середній'"]
ступінь КТ після очищення: [('середній', 18), ('важкий', 10), ('легкий', 7), ('вкрай важкий', 4), ('критичний', 3)]
Пар без пропусків: 42 з 121 (ступінь КТ заповнено лише у 42 пацієнтів)



ступінь КТ,легкий,середній,важкий,вкрай важкий,критичний
Перебіг,,,,,
серед важ,6,10,1,0,0
важкий,1,8,9,3,3
вкрай важкий,0,0,0,1,0


In [14]:
ОБРАНА_МІРА = 'рангова кореляція Спірмена'
ЧОМУ_САМЕ_ВОНА = ('Обидві змінні порядкові (порядкова × порядкова): є природний порядок тяжкості, '
                  'але відстані між рівнями невідомі. Спірмен використовує лише ранги, тому не '
                  'залежить від того, якими числами закодовано рівні; Пірсон вимагає інтервалів, '
                  'а χ²/Крамер викинули б інформацію про порядок.')

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

ρ, p = stats.spearmanr(пара[A].cat.codes, пара[B].cat.codes)
τ, p_τ = stats.kendalltau(пара[A].cat.codes, пара[B].cat.codes)   # контроль: багато зв'язаних рангів
print(f'Спірмен  ρ = {ρ:.3f}   p = {p:.2g}   (n = {len(пара)})')
print(f'Кендалл τ_b = {τ:.3f}   p = {p_τ:.2g}   — перевірка, бо рівнів мало й багато однакових рангів')
print()
print('Висновок: помірно-сильний додатний монотонний зв’язок — що важчий клінічний перебіг,')
print('то вищий ступінь ураження легень за КТ. Зв’язок статистично значущий (p < 0.001),')
print(f'але оцінено лише на {len(пара)} пацієнтах, у яких записано ступінь КТ, а рівень')
print('«вкрай важкий» перебіг представлено одним пацієнтом.')

Спірмен  ρ = 0.630   p = 7.8e-06   (n = 42)
Кендалл τ_b = 0.576   p = 4.7e-05   — перевірка, бо рівнів мало й багато однакових рангів

Висновок: помірно-сильний додатний монотонний зв’язок — що важчий клінічний перебіг,
то вищий ступінь ураження легень за КТ. Зв’язок статистично значущий (p < 0.001),
але оцінено лише на 42 пацієнтах, у яких записано ступінь КТ, а рівень
«вкрай важкий» перебіг представлено одним пацієнтом.


In [15]:
# Контрприклад: Пірсон, застосований «як pandas дозволяє»
# (1) коди категорій за алфавітом — саме так їх видасть .astype('category')
алф_A = df[A].astype('category').cat.codes
алф_B = df[B].astype('category').cat.codes.replace(-1, np.nan)
наївна = pd.concat([алф_A, алф_B], axis=1).dropna()
r_алф = stats.pearsonr(наївна[0], наївна[1])[0]
print('Алфавітний порядок Перебіг:', list(df[A].astype('category').cat.categories))
print(f'(1) Пірсон на алфавітних кодах сирих даних      r = {r_алф:.3f}')

# (2) правильний порядок, але коди 1,2,3,4,5 — рівні відстані, яких ніхто не міряв
r_рівні = stats.pearsonr(пара[A].cat.codes, пара[B].cat.codes)[0]
print(f'(2) Пірсон на кодах 1..5 (рівні кроки)          r = {r_рівні:.3f}')

# (3) той самий порядок, інші відстані: «вкрай важкий» = 10, «критичний» = 20
розтяг = {'легкий': 1, 'середній': 2, 'важкий': 3, 'вкрай важкий': 10, 'критичний': 20}
кт_розтяг = пара[B].astype(str).map(розтяг)
r_розтяг = stats.pearsonr(пара[A].cat.codes, кт_розтяг)[0]
ρ_розтяг = stats.spearmanr(пара[A].cat.codes, кт_розтяг)[0]
print(f'(3) Пірсон на кодах 1,2,3,10,20                 r = {r_розтяг:.3f}')
print()
print(f'Спірмен для (2) і (3): {ρ:.3f} і {ρ_розтяг:.3f} — не змінився.')
print()
print('Чому число Пірсона не можна інтерпретувати: для тих самих пацієнтів і того самого')
print('порядку воно коливається від', round(min(r_алф, r_рівні, r_розтяг), 2), 'до',
      round(max(r_алф, r_рівні, r_розтяг), 2), '— залежно лише від довільно обраних кодів.')
print('Алфавітні коди ще й ламають порядок («вкрай важкий» опиняється між «важкий» і')
print('«серед важ»; «важкий »/«важкий» та 4 написання «середнього» стають різними рівнями).')
print('Пірсон вимірює ЛІНІЙНИЙ зв’язок чисел, а відстаней між рівнями тяжкості не існує.')

Алфавітний порядок Перебіг: ['важкий', 'вкрай важкий ', 'серед важ']
(1) Пірсон на алфавітних кодах сирих даних      r = 0.424
(2) Пірсон на кодах 1..5 (рівні кроки)          r = 0.596
(3) Пірсон на кодах 1,2,3,10,20                 r = 0.418

Спірмен для (2) і (3): 0.630 і 0.630 — не змінився.

Чому число Пірсона не можна інтерпретувати: для тих самих пацієнтів і того самого
порядку воно коливається від 0.42 до 0.6 — залежно лише від довільно обраних кодів.
Алфавітні коди ще й ламають порядок («вкрай важкий» опиняється між «важкий» і
«серед важ»; «важкий »/«важкий» та 4 написання «середнього» стають різними рівнями).
Пірсон вимірює ЛІНІЙНИЙ зв’язок чисел, а відстаней між рівнями тяжкості не існує.


---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [16]:
ПІДГОТОВКА = {
    'номер і/х':    'не беремо в модель — ідентифікатор, не величина; лише для пошуку дублікатів',
    'помер/жив':    'one-hot / один індикатор «помер» = 1, «вижив» = 0 (бінарна номінальна, це зазвичай ціль)',
    'Перебіг':      'впорядкована категорія з кодами 0/1/2 за порядком, заданим вручну: серед важ < важкий < вкрай важкий',
    'Виписка':      'дату саму не подаємо; переходимо до тривалості (Виписка − Поступлення, днів) і стандартизуємо',
    'spO2 без дот': 'виправити 0.9 → 90, далі стандартизація (z-score); можлива нормалізація min-max у [0, 1]',
    'Креат1':       'перетворити рядки на числа, логарифмувати (правобічна асиметрія, max 281.7) і стандартизувати',
    'Гемогл2':      'перетворити «немає» на NaN, заповнити медіаною або позначити пропуск, стандартизувати',
    'БРзаг2':       'логарифмування (викид 60.5) і стандартизація; пропуски (20) — окремий індикатор',
}
for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,номер і/х,ідентифікатор,"не беремо в модель — ідентифікатор, не величина; лише для пошуку дублікатів"
1,помер/жив,номінальна,"one-hot / один індикатор «помер» = 1, «вижив» = 0 (бінарна номінальна, це зазвичай ціль)"
2,Перебіг,порядкова,"впорядкована категорія з кодами 0/1/2 за порядком, заданим вручну: серед важ < важкий < вкрай важкий"
3,Виписка,інтервальна,"дату саму не подаємо; переходимо до тривалості (Виписка − Поступлення, днів) і стандартизуємо"
4,spO2 без дот,абсолютна,"виправити 0.9 → 90, далі стандартизація (z-score); можлива нормалізація min-max у [0, 1]"
5,Креат1,відношень,"перетворити рядки на числа, логарифмувати (правобічна асиметрія, max 281.7) і стандартизувати"
6,Гемогл2,відношень,"перетворити «немає» на NaN, заповнити медіаною або позначити пропуск, стандартизувати"
7,БРзаг2,відношень,логарифмування (викид 60.5) і стандартизація; пропуски (20) — окремий індикатор


In [17]:
def нормалізувати(s):
    """Min-max у [0, 1] з перевіркою варіативності замість мовчазного NaN."""
    s = pd.to_numeric(s, errors='coerce')
    розмах = s.max() - s.min()
    if not розмах > 0:
        raise ValueError(f'{s.name}: сталий стовпець (max = min), нормалізувати неможливо')
    return (s - s.min()) / розмах


def стандартизувати(s):
    s = pd.to_numeric(s, errors='coerce')
    if not s.std() > 0:
        raise ValueError(f'{s.name}: нульове стандартне відхилення')
    return (s - s.mean()) / s.std()


підготовлені = pd.DataFrame(index=df.index)

# 1) номінальна: помер/жив -> індикатор (one-hot без зайвого стовпця для бінарної ознаки)
підготовлені['помер'] = (df['помер/жив'] == 1).astype(int)

# 2) порядкова: Перебіг -> цілі коди за ЗАДАНИМ порядком (не алфавітним)
підготовлені['перебіг_код'] = pd.Series(очищений('Перебіг'), index=df.index).cat.codes

# 3) інтервальна: Виписка -> тривалість перебування, потім стандартизація
днів = (очищений('Виписка') - df['Поступлення']).dt.days
днів = днів.where(днів >= 0)            # 2 записи з випискою раніше за поступлення — помилки дат
підготовлені['тривалість_z'] = стандартизувати(днів.rename('тривалість'))

# 4) відношень: Креат1 -> log, потім min-max
підготовлені['креат1_log_norm'] = нормалізувати(np.log(очищений('Креат1')).rename('Креат1'))

# 5) абсолютна: spO2 -> min-max
підготовлені['spO2_norm'] = нормалізувати(очищений('spO2 без дот').rename('spO2'))

print('Перебіг -> код:', dict(zip(ПОРЯДОК_ПЕРЕБІГ, range(3))))
print(f'Від’ємна тривалість (помилки дат) відкинуто: {(( очищений("Виписка") - df["Поступлення"]).dt.days < 0).sum()}')
print()
print(підготовлені.describe().round(3).to_string())
print()
# Що буде зі сталим стовпцем (ФЕРИТ2 має одне значення з 121):
try:
    нормалізувати(df['ФЕРИТ2'])
except ValueError as e:
    print('Перевірка на сталому стовпці спрацювала:', e)

Перебіг -> код: {'серед важ': 0, 'важкий': 1, 'вкрай важкий': 2}
Від’ємна тривалість (помилки дат) відкинуто: 2

         помер  перебіг_код  тривалість_z  креат1_log_norm  spO2_norm
count  121.000      121.000       116.000          116.000    120.000
mean     0.504        0.760        -0.000            0.366      0.683
std      0.502        0.448         1.000            0.184      0.202
min      0.000        0.000        -0.844            0.000      0.000
25%      0.000        1.000        -0.462            0.225      0.620
50%      1.000        1.000        -0.174            0.351      0.708
75%      1.000        1.000         0.125            0.491      0.818
max      1.000        2.000         8.343            1.000      1.000

Перевірка на сталому стовпці спрацювала: ФЕРИТ2: сталий стовпець (max = min), нормалізувати неможливо


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [18]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [19]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [20]:
ВИПРАВЛЕННЯ = [
    (1, 'Середнє обчислено для номінальної шкали (помер/жив); 1 і 2 — мітки, а не величини.',
        '1.50 не означає «посередині між смертю й одужанням»: такого стану не існує. При кодуванні 0/1 '
        'вийшло б 0.50, при 1/0 — 0.50 у протилежному сенсі; число залежить лише від вибору кодів.',
        'Мода й частоти: помер 61 (50.4 %), вижив 60 (49.6 %) — летальність 50.4 %.'),
    (2, 'Дані непридатні: одна номінальна категорія записана чотирма написаннями (ж/жін, ч/чол).',
        'nunique = 4 рахує варіанти написання, а не статі; «4 статі» та «найбільша група ж (46 %)» — '
        'артефакт введення, реальна частка жінок інша.',
        'Нормалізувати написання (перша літера після strip) і рахувати частоти: ж 75 (62 %), ч 46 (38 %).'),
    (3, 'Порядкова шкала: порядок узято алфавітний, написання не уніфіковано, а потім пораховано середнє кодів.',
        'Алфавіт ставить «важкий» найлегшим, а «середній» найважчим; 12 «рівнів» замість 5; середнє 5.9 '
        'кодів без відстаней — порядкова шкала середнього не допускає, тому «середній ступінь — легкий» хибне.',
        'Уніфікувати написання, задати порядок вручну (легкий<середній<важкий<вкрай важкий<критичний) і '
        'рахувати медіану/моду: медіана — «середній» (n = 42).'),
    (4, 'Дані непридатні: to_numeric(errors=coerce) мовчки викинув 17 текстових записів — 16 діапазонів («70-80%», «до 17%») і рядок із пробілів.',
        '39.6 % пораховано лише на 27 із 43 реально заповнених КТ; відкинуті діапазони здебільшого великі (60–80 %), '
        'тож середнє зміщене донизу, і це не «середній обсяг ураження» відділення.',
        'Розібрати діапазони (середина інтервалу, «до 17%» → верхня межа), привести до однієї одиниці, '
        'вказати n; для асиметричних даних — медіана з IQR.'),
    (5, 'Дані непридатні: значення 0.9 — сатурація, записана часткою (90 %), а не 0.9 %.',
        'Мінімум 0.9 фізіологічно неможливий, середнє 82.05 занижене через одну помилку; крім того, середнє '
        'не каже нічого про «більшість» пацієнтів.',
        'Виправити 0.9 → 90 (або відкинути), повідомляти медіану 84 % з IQR (79.75–89.25 %) і частку spO2 < 90 %.'),
    (6, 'Інтервальна шкала: коефіцієнт варіації (std/mean) обчислено для дат, у яких нуль умовний (1970-01-01).',
        '0.595 % — це розкид дат відносно відстані від 1970 року; перенісши нуль на 2021-01-01, отримаємо '
        'зовсім інше число. Однорідності потоку госпіталізацій воно не вимірює.',
        'Відмовитися від CV; описати розкид у днях (std ≈ 113 днів, розмах), або CV інтервалів між поступленнями.'),
    (7, 'Номінальна × номінальна: Пірсон на кодах статі та смерті; висновок «близько нуля» хибний.',
        'r = 0.177 — це φ для таблиці 2×2, а не «відсутність лінійного зв’язку»: лінійності для категорій '
        'не існує, і 0.177 — не нуль; значущість не перевірено.',
        'Таблиця спряженості, χ² і коефіцієнт Крамера: χ² = 3.78, p ≈ 0.052, V = 0.18 — слабкий зв’язок на межі значущості.'),
    (8, 'Min-max застосовано до всіх числових стовпців, включно з номінальним помер/жив і майже порожнім/сталим ФЕРИТ2.',
        'ФЕРИТ2 має одне значення: max − min = 0, тож замість [0, 1] отримано мовчазні NaN; «усі ознаки '
        'нормалізовано» — неправда, а нормалізована мітка помер/жив не є величиною.',
        'Перед нормалізацією перевіряти розмах > 0, виключати сталі й ідентифікаційні/номінальні стовпці, '
        'номінальні кодувати one-hot.'),
    (9, 'Дані непридатні для dropna(): кожен рядок має пропуск хоча б в одному з 77 стовпців (ФЕРИТ2 — 120 NaN).',
        'Залишилось 0 рядків із 121, а висновок стверджує «повністю заповнений набір, придатний для моделі» — '
        'насправді даних для моделі немає взагалі.',
        'Видаляти пропуски лише в потрібних для аналізу стовпцях (dropna(subset=...)), відкинути стовпці з >50 % NaN, '
        'решту імпутувати.'),
    (10, 'Витік цільової змінної + інтервальна шкала: дата розтину існує лише для померлих, NaT стає числом −9.2·10¹⁸.',
        'r = 1.00 означає лише «розтин роблять тим, хто помер» — ознака є наслідком результату, а не предиктором; '
        'на момент госпіталізації її немає. Кореляція Пірсона з бінарною ціллю для дати ще й недопустима.',
        'Виключити Дату розтину (і Виписку) з ознак; для числових ознак × бінарна ціль — точковий бісеріальний '
        'коефіцієнт на очищених даних.'),
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,"Середнє обчислено для номінальної шкали (помер/жив); 1 і 2 — мітки, а не величини.","1.50 не означає «посередині між смертю й одужанням»: такого стану не існує. При кодуванні 0/1 вийшло б 0.50, при 1/0 — 0.50 у протилежному сенсі; число залежить лише від вибору кодів.","Мода й частоти: помер 61 (50.4 %), вижив 60 (49.6 %) — летальність 50.4 %."
1,2,"Дані непридатні: одна номінальна категорія записана чотирма написаннями (ж/жін, ч/чол).","nunique = 4 рахує варіанти написання, а не статі; «4 статі» та «найбільша група ж (46 %)» — артефакт введення, реальна частка жінок інша.","Нормалізувати написання (перша літера після strip) і рахувати частоти: ж 75 (62 %), ч 46 (38 %)."
2,3,"Порядкова шкала: порядок узято алфавітний, написання не уніфіковано, а потім пораховано середнє кодів.","Алфавіт ставить «важкий» найлегшим, а «середній» найважчим; 12 «рівнів» замість 5; середнє 5.9 кодів без відстаней — порядкова шкала середнього не допускає, тому «середній ступінь — легкий» хибне.","Уніфікувати написання, задати порядок вручну (легкий<середній<важкий<вкрай важкий<критичний) і рахувати медіану/моду: медіана — «середній» (n = 42)."
3,4,"Дані непридатні: to_numeric(errors=coerce) мовчки викинув 17 текстових записів — 16 діапазонів («70-80%», «до 17%») і рядок із пробілів.","39.6 % пораховано лише на 27 із 43 реально заповнених КТ; відкинуті діапазони здебільшого великі (60–80 %), тож середнє зміщене донизу, і це не «середній обсяг ураження» відділення.","Розібрати діапазони (середина інтервалу, «до 17%» → верхня межа), привести до однієї одиниці, вказати n; для асиметричних даних — медіана з IQR."
4,5,"Дані непридатні: значення 0.9 — сатурація, записана часткою (90 %), а не 0.9 %.","Мінімум 0.9 фізіологічно неможливий, середнє 82.05 занижене через одну помилку; крім того, середнє не каже нічого про «більшість» пацієнтів.","Виправити 0.9 → 90 (або відкинути), повідомляти медіану 84 % з IQR (79.75–89.25 %) і частку spO2 < 90 %."
5,6,"Інтервальна шкала: коефіцієнт варіації (std/mean) обчислено для дат, у яких нуль умовний (1970-01-01).","0.595 % — це розкид дат відносно відстані від 1970 року; перенісши нуль на 2021-01-01, отримаємо зовсім інше число. Однорідності потоку госпіталізацій воно не вимірює.","Відмовитися від CV; описати розкид у днях (std ≈ 113 днів, розмах), або CV інтервалів між поступленнями."
6,7,Номінальна × номінальна: Пірсон на кодах статі та смерті; висновок «близько нуля» хибний.,"r = 0.177 — це φ для таблиці 2×2, а не «відсутність лінійного зв’язку»: лінійності для категорій не існує, і 0.177 — не нуль; значущість не перевірено.","Таблиця спряженості, χ² і коефіцієнт Крамера: χ² = 3.78, p ≈ 0.052, V = 0.18 — слабкий зв’язок на межі значущості."
7,8,"Min-max застосовано до всіх числових стовпців, включно з номінальним помер/жив і майже порожнім/сталим ФЕРИТ2.","ФЕРИТ2 має одне значення: max − min = 0, тож замість [0, 1] отримано мовчазні NaN; «усі ознаки нормалізовано» — неправда, а нормалізована мітка помер/жив не є величиною.","Перед нормалізацією перевіряти розмах > 0, виключати сталі й ідентифікаційні/номінальні стовпці, номінальні кодувати one-hot."
8,9,Дані непридатні для dropna(): кожен рядок має пропуск хоча б в одному з 77 стовпців (ФЕРИТ2 — 120 NaN).,"Залишилось 0 рядків із 121, а висновок стверджує «повністю заповнений набір, придатний для моделі» — насправді даних для моделі немає взагалі.","Видаляти пропуски лише в потрібних для аналізу стовпцях (dropna(subset=...)), відкинути стовпці з >50 % NaN, решту імпутувати."
9,10,"Витік цільової змінної + інтервальна шкала: дата розтину існує лише для померлих, NaT стає числом −9.2·10¹⁸.","r = 1.00 означає лише «розтин роблять тим, хто помер» — ознака є наслідком результату, а не предиктором; на момент госпіталізації її немає. Кореляція Пірсона з бінарною ціллю для дати ще й недопустима.",Виключити Дату розтину (і Виписку) з ознак; для числових ознак × бінарна ціль — точковий бісеріальний коефіцієнт на очищен

### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [21]:
# --- Розділ 1: середнє номінальної ознаки -> частоти
print('РОЗДІЛ 1')
print('  неправильно: середнє помер/жив =', round(df['помер/жив'].mean(), 2))
частки = df['помер/жив'].map({1: 'помер', 2: 'вижив'}).value_counts()
print('  правильно  :', частки.to_dict(),
      f'-> летальність {100 * частки["помер"] / частки.sum():.1f} %')
print()

# --- Розділ 3: алфавітний порядок + середнє кодів -> заданий порядок + медіана
print('РОЗДІЛ 3')
сирі = df['ступінь КТ'].value_counts().sort_index()
коди_алф = {н: i for i, н in enumerate(сирі.index)}
сер = df['ступінь КТ'].map(коди_алф).mean()
print('  неправильно: «найлегший» =', сирі.index[0], '| середнє кодів =', round(сер, 2),
      '->', сирі.index[int(round(сер))])
print('  правильно  : рівні за заданим порядком:', list(кт.value_counts(sort=False).items()))
print('               медіана =', ПОРЯДОК_КТ[int(кт.dropna().cat.codes.median())],
      '| мода =', кт.mode()[0], f'| n = {кт.notna().sum()}')
print()

# --- Розділ 6: CV дат залежить від вибору нуля -> розкид у днях
print('РОЗДІЛ 6')
дати = pd.to_datetime(df['Поступлення'], errors='coerce').dropna()
від_1970 = (дати - pd.Timestamp('1970-01-01')).dt.days
від_2021 = (дати - pd.Timestamp('2021-01-01')).dt.days
print(f'  неправильно: CV від 1970-01-01 = {100 * від_1970.std() / від_1970.mean():.3f} %')
print(f'               CV від 2021-01-01 = {100 * від_2021.std() / від_2021.mean():.1f} %   <- ті самі дати, інший «нуль»')
print(f'  правильно  : стд. відхилення = {від_1970.std():.0f} днів, '
      f'розмах {дати.min().date()} — {дати.max().date()} ({(дати.max() - дати.min()).days} днів)')
print()

# --- Розділ 7: Пірсон стать × летальність -> χ² і Крамер
print('РОЗДІЛ 7')
стать = df['Стать'].astype(str).str.strip().str[0]
помер = (df['помер/жив'] == 1).astype(int)
print('  неправильно: Пірсон r =', round(стать.map({'ч': 0, 'ж': 1}).corr(помер), 3), '-> «зв’язку немає»')
таблиця = pd.crosstab(стать, df['помер/жив'].map({1: 'помер', 2: 'вижив'}))
χ2, p_χ2, dof, _ = stats.chi2_contingency(таблиця, correction=False)
V = np.sqrt(χ2 / (таблиця.values.sum() * (min(таблиця.shape) - 1)))
print('  правильно  :'); print(таблиця.to_string())
print(f'               χ² = {χ2:.2f}, p = {p_χ2:.3f}, V Крамера = {V:.3f}')
print('               -> слабкий зв’язок на межі значущості (летальність жінок 57 %, чоловіків 39 %)')
print('                  — висновок «зв’язку немає» не обґрунтований.')

РОЗДІЛ 1
  неправильно: середнє помер/жив = 1.5
  правильно  : {'помер': 61, 'вижив': 60} -> летальність 50.4 %

РОЗДІЛ 3
  неправильно: «найлегший» = важкий | середнє кодів = 5.9 -> легкий
  правильно  : рівні за заданим порядком: [('легкий', 7), ('середній', 18), ('важкий', 10), ('вкрай важкий', 4), ('критичний', 3)]
               медіана = середній | мода = середній | n = 42

РОЗДІЛ 6
  неправильно: CV від 1970-01-01 = 0.595 %
               CV від 2021-01-01 = 30.0 %   <- ті самі дати, інший «нуль»
  правильно  : стд. відхилення = 113 днів, розмах 2021-02-25 — 2022-06-03 (463 днів)

РОЗДІЛ 7
  неправильно: Пірсон r = 0.177 -> «зв’язку немає»
  правильно  :
помер/жив  вижив  помер
Стать                  
ж             32     43
ч             28     18
               χ² = 3.78, p = 0.052, V Крамера = 0.177
               -> слабкий зв’язок на межі значущості (летальність жінок 57 %, чоловіків 39 %)
                  — висновок «зв’язку немає» не обґрунтований.


---
# Етап 7. Висновки та індивідуальне питання

In [22]:
ВИСНОВКИ = [
    'У стовпці «помер/жив» 61 летальний випадок і 60 виживань (летальність 50.4 %); усі 61 смерть '
    'припадають на пацієнтів із «Перебіг» = «важкий» (летальність у групі 68 %), а серед «серед важ» '
    '(30 осіб) не помер ніхто; оскільки «помер/жив» номінальна, зв’язок описую таблицею частот, а не середнім кодом.',

    'Ступінь КТ і клінічний «Перебіг» пов’язані монотонно: ρ Спірмена = 0.63 (p < 0.001, n = 42); '
    'при цьому Пірсон на тих самих даних дає від 0.42 до 0.60 залежно від довільного кодування рівнів, '
    'що й доводить його непридатність для порядкових шкал.',

    'Лабораторні стовпці мого варіанта мають дефекти, які describe() не показує: «Креат1» і «Гемогл2» '
    'зберігаються як object через рядки «даних», «немає», «109.9»; «БРзаг2» приймає лише 21 значення, '
    'і 89 % записів — кратні 1.7 (11.9, 13.6, 15.3, 17.0, 18.7), тобто значення перераховані з мг/дл із грубим кроком.',

    'У «spO2 без дот» одне значення 0.9 записане часткою замість 90 %: після виправлення середнє зростає '
    'з 82.05 до 82.79, а std падає з 12.2 до 9.7; медіана (84 %) стійка до цієї помилки. Медіанна '
    'сатурація померлих — 80 %, тих, хто вижив, — 89 %.',

    'У «Виписка» дві дати неіснуючі («24.024.2021», «20.24.2021»), у двох записах виписка раніша за '
    'поступлення (до −29 днів), а в одному перебування триває 194 дні — інтервальна шкала дозволяє рахувати тривалості, але лише після перевірки, що вони невід’ємні.',
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. У стовпці «помер/жив» 61 летальний випадок і 60 виживань (летальність 50.4 %); усі 61 смерть припадають на пацієнтів із «Перебіг» = «важкий» (летальність у групі 68 %), а серед «серед важ» (30 осіб) не помер ніхто; оскільки «помер/жив» номінальна, зв’язок описую таблицею частот, а не середнім кодом.
2. Ступінь КТ і клінічний «Перебіг» пов’язані монотонно: ρ Спірмена = 0.63 (p < 0.001, n = 42); при цьому Пірсон на тих самих даних дає від 0.42 до 0.60 залежно від довільного кодування рівнів, що й доводить його непридатність для порядкових шкал.
3. Лабораторні стовпці мого варіанта мають дефекти, які describe() не показує: «Креат1» і «Гемогл2» зберігаються як object через рядки «даних», «немає», «109.9»; «БРзаг2» приймає лише 21 значення, і 89 % записів — кратні 1.7 (11.9, 13.6, 15.3, 17.0, 18.7), тобто значення перераховані з мг/дл із грубим кроком.
4. У «spO2 без дот» одне значення 0.9 записане часткою замість 90 %: після виправлення середнє зростає з 82.05 до 82.79, а std падає з 12

In [23]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()

пж = df['помер/жив']
print(f'mean = {пж.mean():.3f}   std = {пж.std():.3f}   median = {пж.median():.0f}   '
      f'25% = {пж.quantile(.25):.0f}   75% = {пж.quantile(.75):.0f}')
інше = пж.map({1: 2, 2: 1})        # ті самі пацієнти, лише мітки поміняно місцями
print(f'після обміну міток 1<->2: mean = {інше.mean():.3f}   median = {інше.median():.0f}   '
      f'std = {інше.std():.3f}')
print()

ВІДПОВІДЬ = '''
«помер/жив» — номінальна бінарна ознака: 1 і 2 лише назви категорій. Тип int64 змушує pandas
вважати стовпець числовим, тож describe() без жодного попередження видає, зокрема, три числа:

1) Середнє = 1.496. Його «прочитали» б як «пацієнт наполовину помер». Насправді це лише
   1 + частка міток «2»: при кодуванні 0/1 вийшло б 0.496, при 10/20 — 14.96. Число
   визначає вибір кодів, а не результат лікування. Змістовна статистика — частоти й мода:
   61 помер (50.4 %), 60 вижили.

2) Стандартне відхилення = 0.502. Воно вимірює розкид відстаней від «середнього», але
   відстані між «помер» і «вижив» не існує; при кодуванні 1/10 std стала б у 9 разів
   більшою за тих самих пацієнтів. Для номінальної шкали міра розкиду — частоти (або
   ентропія), не std.

3) Медіана = 1 (та квартилі 25 % = 1, 75 % = 2). Медіана вимагає порядку, а «смерть менша
   за життя» — не властивість даних, а випадок кодування. Якщо поміняти мітки місцями,
   медіана стане 2 — «медіанний пацієнт вижив», хоча жоден пацієнт не змінився. Крім того,
   при 61/60 вона залежить від одного пацієнта.

pandas не скаржиться, бо бачить лише dtype: шкали вимірювання він не знає. Правильно — або
перетворити стовпець на category, тоді describe() видасть count/unique/top/freq, або
відразу перекодувати в індикатор «помер» (0/1), де середнє вже має зміст: це частка
померлих, 0.504.
'''
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Стовпець «помер/жив» містить числа 1 і 2. Які три різні статистики pandas порахує для нього без жодної скарги — і чому кожна з них безглузда?

mean = 1.496   std = 0.502   median = 1   25% = 1   75% = 2
після обміну міток 1<->2: mean = 1.504   median = 2   std = 0.502


«помер/жив» — номінальна бінарна ознака: 1 і 2 лише назви категорій. Тип int64 змушує pandas
вважати стовпець числовим, тож describe() без жодного попередження видає, зокрема, три числа:

1) Середнє = 1.496. Його «прочитали» б як «пацієнт наполовину помер». Насправді це лише
   1 + частка міток «2»: при кодуванні 0/1 вийшло б 0.496, при 10/20 — 14.96. Число
   визначає вибір кодів, а не результат лікування. Змістовна статистика — частоти й мода:
   61 помер (50.4 %), 60 вижили.

2) Стандартне відхилення = 0.502. Воно вимірює розкид відстаней від «середнього», але
   відстані між «помер» і «вижив» не існує; при кодуванні 1/10 std стала б у 9 разів
   більшою за тих самих пацієнтів. Для номінальної шкали мір

---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [24]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'текст, як є',
    'етап 2 — допустимі статистики':   'код, перевірено',
    'етап 3 — безглузді числа':        'текст, як є',
    'етап 4 — міра зв’язку':           'код, перевірено',
    'етап 5 — підготовка до моделі':   'код, перевірено',
    'етап 6 — виправлення аналізу':    'текст, як є',
    'етап 7 — висновки й відповідь':   'текст, як є',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = (
    'Перезапустила ноутбук на чистому ядрі (Restart & Run All). Перевірила вручну: сирі значення '
    'стовпців (рядки «даних», «немає», дату «24.024.2021», spO2 = 0.9); що порядок Перебігу й ступеня КТ '
    'задано вручну, а не за алфавітом; що Спірмен не змінюється при зміні кодів, а Пірсон змінюється; '
    'що нормалізація сталого ФЕРИТ2 дає помилку, а не NaN; числа в розділах 1, 3, 6, 7 помилкового звіту.'
)
ПІДТВЕРДЖУЮ = True      # я можу пояснити кожен рядок свого коду
                        # і внести в нього зміни на вимогу під час захисту

In [25]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 3 готова до захисту.
Бобринок Ангеліна, ШІ-32


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.